# Aprendizado de Máquina — Aula prática 11

## KNN e Árvores de Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Esta é a última aula do curso regular, e ela fecha um circuito. Os dois métodos não
paramétricos que estudamos em **regressão** — o KNN da Aula 04 e as árvores da Aula
06 — reaparecem aqui em **classificação**. A estrutura é a mesma; o que muda é o
critério que decide uma divisão, e é aí que mora o conteúdo novo:

> **em classificação, "erro de treino" é uma medida ruim para *crescer* uma árvore
> — mesmo sendo a medida certa para *avaliá-la*.**

Vamos exibir um caso em que o erro de classificação diz que uma divisão útil é
inútil, ver por que Gini e entropia não têm esse defeito, e terminar com todos os
classificadores do bloco lado a lado nos mesmos dados — que é a melhor maneira de
fixar a intuição de "flexível × rígido" que percorreu o curso inteiro.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- ver o $k$ do KNN funcionando como botão de flexibilidade em classificação;
- calcular Gini, entropia e erro de classificação, e **exibir o contraexemplo** em
  que o erro não decresce numa divisão claramente boa;
- comparar árvores crescidas com `criterion="gini"` e `"entropy"`;
- lembrar que `max_features` tem padrões **diferentes** no classificador e no
  regressor;
- comparar floresta e AdaBoost com as métricas da Aula 09.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Nada de novo, a rigor: são as versões classificadoras dos estimadores que já
usamos em regressão, mais o `AdaBoostClassifier`.

In [ ]:
import sklearn.model_selection as skm
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def desenhar(ax, modelo, X, y, titulo=""):
    gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 300),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 300))
    Z = modelo.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, Z, levels=[-0.5, 0.5, 1.5],
                colors=["steelblue", "crimson"], alpha=0.15)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=12, color="steelblue", edgecolor="k", lw=0.2)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=12, color="crimson", edgecolor="k", lw=0.2)
    ax.set_title(titulo, fontsize=9); ax.set_xticks([]); ax.set_yticks([])

---
## 2. KNN em classificação: o mesmo botão

Em regressão, o KNN tirava a **média** das respostas dos $k$ vizinhos. Em
classificação ele tira a **proporção** — que é a estimativa plug-in de
$P(Y=1\mid x)$ — e decide pela maioria. É literalmente a mesma conta.

In [ ]:
X, y = make_moons(n_samples=400, noise=0.30, random_state=3)
X_te, y_te = make_moons(n_samples=8000, noise=0.30, random_state=4)

fig, axes = subplots(1, 4, figsize=(11, 2.9))
for ax, k in zip(axes, [1, 5, 25, 150]):
    m = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    desenhar(ax, m, X, y,
             f"k = {k}\ntreino {m.score(X, y):.3f}  teste {m.score(X_te, y_te):.3f}")

Com $k=1$ a fronteira é um mosaico que envolve cada ponto isolado — acurácia
perfeita no treino e nada de generalização. Com $k=150$ a fronteira quase não
distingue os dois arcos. É a mesma figura da Aula 04, com classes no lugar de uma
curva.

E `predict_proba` aqui devolve literalmente a fração de vizinhos de cada classe:

In [ ]:
m5 = KNeighborsClassifier(n_neighbors=5).fit(X, y)
# um ponto perto da fronteira, onde os vizinhos discordam
proba = m5.predict_proba(X_te)[:, 1]
ponto = X_te[[np.argmin(np.abs(proba - 0.5))]]
dist, viz = m5.kneighbors(ponto)
print(f"classes dos 5 vizinhos mais proximos: {y[viz[0]]}")
print(f"fracao da classe 1                  : {y[viz[0]].mean():.2f}")
print(f"predict_proba                       : {m5.predict_proba(ponto)[0, 1]:.2f}")
print(f"\nprobabilidades possiveis com k=5: {sorted(set(np.round(np.arange(6)/5, 2)))}")

Note a consequência: com $k=5$ a probabilidade estimada só pode assumir **seis
valores**. Um classificador cujas probabilidades vivem numa grade grosseira é um
mau candidato a qualquer conta de custo esperado (Aula 09, Seção 5) — e $k$ passa a
ter uma segunda razão para ser grande, além do balanço viés–variância.

---
## 3. Três medidas de impureza, e por que uma delas não serve

Para dividir um nó, a árvore de regressão minimizava a soma de quadrados. Em
classificação, os três candidatos naturais para um nó com proporção $p$ da classe 1
são

$$\text{erro} = 1 - \max(p, 1-p), \qquad
  \text{Gini} = 2p(1-p), \qquad
  \text{entropia} = -p\log_2 p - (1-p)\log_2(1-p).$$

Os três valem zero num nó puro e são máximos em $p = 1/2$. Parecem
intercambiáveis. Não são.

In [ ]:
def erro(p):
    return 1 - np.maximum(p, 1 - p)


def gini(p):
    return 2 * p * (1 - p)


def entropia(p):
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
    return np.nan_to_num(h)


pp = np.linspace(0, 1, 400)
fig, ax = subplots(figsize=(5.2, 3.2))
ax.plot(pp, erro(pp), label="erro de classificacao", lw=1.8)
ax.plot(pp, gini(pp), label="Gini", lw=1.8)
ax.plot(pp, entropia(pp) / 2, label="entropia (dividida por 2)", lw=1.8)
ax.set_xlabel("p (proporcao da classe 1 no no)"); ax.set_ylabel("impureza")
ax.legend(fontsize=8)

A diferença está na **curvatura**. O erro de classificação é formado por dois
segmentos de reta; Gini e entropia são estritamente côncavas. E é a concavidade
estrita que garante que dividir um nó **sempre** reduz a impureza — a menos que os
dois filhos tenham exatamente a mesma proporção.

Com o erro de classificação isso falha, e o contraexemplo é aritmético.

In [ ]:
# pai: 400 observacoes, 300 da classe 1  -> p = 0,75
n_pai, p_pai = 400, 0.75
# filhos: 200 com p=0,50 e 200 com p=1,00
filhos = [(200, 0.50), (200, 1.00)]

for nome, f in [("erro", erro), ("Gini", gini), ("entropia", entropia)]:
    antes = f(p_pai)
    depois = sum(n * f(p) for n, p in filhos) / n_pai
    print(f"{nome:9s}: pai = {antes:.4f}   filhos (media ponderada) = {depois:.4f}"
          f"   reducao = {antes - depois:+.4f}")

> **A lição.** A divisão é claramente útil: ela produz um nó **puro** com 200
> observações da classe 1. Gini e entropia registram uma redução; o erro de
> classificação registra redução **exatamente zero**.
>
> Uma árvore que crescesse minimizando o erro de classificação recusaria essa
> divisão, pelo mesmo motivo que o algoritmo ganancioso da Aula 06 recusava o
> primeiro corte do XOR — e, de novo, o problema não é o critério estar errado, é
> ele ser cego a um passo de distância.
>
> Daí a divisão de trabalho que o `scikit-learn` adota: **Gini ou entropia para
> crescer, erro de classificação para podar e avaliar.** O `criterion` do
> `DecisionTreeClassifier` só aceita as duas primeiras.

> **Sua vez.** Encontre você mesmo outro par de filhos, com o mesmo pai
> ($n=400$, $p=0{,}75$), em que a redução do erro de classificação seja zero mas a
> do Gini seja positiva. Depois descreva a família inteira desses casos: o que os
> filhos precisam ter em comum?

---
## 4. Gini contra entropia: importa na prática?

Os dois critérios são côncavos e parecidos. A pergunta é se a escolha entre eles
muda alguma coisa. Vamos crescer as duas árvores e comparar, primeiro a estrutura,
depois o desempenho.

In [ ]:
Xb, yb = make_moons(n_samples=1200, noise=0.32, random_state=11)
Xb_te, yb_te = make_moons(n_samples=8000, noise=0.32, random_state=12)

linhas = []
for crit in ["gini", "entropy"]:
    for prof in [3, 6, None]:
        m = DecisionTreeClassifier(criterion=crit, max_depth=prof,
                                   random_state=0).fit(Xb, yb)
        linhas.append({"criterio": crit, "profundidade": str(prof),
                       "folhas": m.get_n_leaves(),
                       "acuracia (teste)": m.score(Xb_te, yb_te)})
pd.DataFrame(linhas).set_index(["criterio", "profundidade"]).round(4)

In [ ]:
g = DecisionTreeClassifier(criterion="gini", max_depth=6, random_state=0).fit(Xb, yb)
e = DecisionTreeClassifier(criterion="entropy", max_depth=6, random_state=0).fit(Xb, yb)
concordam = (g.predict(Xb_te) == e.predict(Xb_te)).mean()

print(f"corte da raiz  -- Gini: x{g.tree_.feature[0]+1} <= {g.tree_.threshold[0]:.3f}")
print(f"corte da raiz  -- entropia: x{e.tree_.feature[0]+1} <= {e.tree_.threshold[0]:.3f}")
print(f"\nas duas arvores concordam em {concordam:.2%} das 8000 observacoes de teste")

As diferenças são pequenas, e é isso que a literatura diz: entre Gini e entropia a
escolha raramente muda o resultado. Elas discordam um pouco mais em nós muito
desbalanceados, onde a entropia — que cresce mais rápido perto de $p=0$ e $p=1$ —
é um pouco mais exigente.

Compare com o que a Aula 06 mediu sobre o `ccp_alpha` e a Aula 04 sobre a janela
$h$: há hiperparâmetros que decidem o resultado e há escolhas que só decidem o
gosto. Saber em qual categoria cada um está é o que separa ajustar um modelo de
girar botões.

---
## 5. Uma pegadinha de configuração: `max_features`

A Aula 06 explicou que a floresta aleatória sorteia $m < d$ covariáveis em cada nó.
A regra empírica é $m \approx d/3$ em regressão e $m \approx \sqrt d$ em
classificação. O `scikit-learn` adota essas regras — mas **só uma delas é o
padrão**.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

print("padrao de max_features:")
print(f"   RandomForestClassifier: {RandomForestClassifier().max_features!r}")
print(f"   RandomForestRegressor : {RandomForestRegressor().max_features!r}")

d = 30
print(f"\ncom d = {d} covariaveis, isso significa:")
print(f"   no classificador: m = sqrt({d}) = {int(np.sqrt(d))}  -> floresta de verdade")
print(f"   no regressor    : m = {d}          -> isso e' BAGGING, nao floresta")

Ou seja: `RandomForestRegressor()`, na configuração de fábrica, **não é uma floresta
aleatória** — é *bagging*. Se você quer a descorrelação que a Aula 06 mediu, precisa
dizer `max_features=1/3` explicitamente. No classificador, o padrão já faz a coisa
certa.

É o tipo de detalhe que não aparece em nenhuma aula teórica e decide resultados.

---
## 6. Todas as fronteiras, lado a lado

Chegou a hora de pôr o bloco inteiro nos mesmos dados. Cada painel é um método das
Aulas 08 a 11, e o eixo que os organiza é o mesmo desde a Aula 01: **quanta
flexibilidade**.

In [ ]:
todos = [
    ("logistica", Pipeline([("sc", StandardScaler()),
                            ("lg", LogisticRegression())])),
    ("KNN (k=1)", KNeighborsClassifier(n_neighbors=1)),
    ("KNN (k=25)", KNeighborsClassifier(n_neighbors=25)),
    ("arvore (prof. 3)", DecisionTreeClassifier(max_depth=3, random_state=0)),
    ("arvore (cheia)", DecisionTreeClassifier(random_state=0)),
    ("floresta (B=300)", RandomForestClassifier(n_estimators=300, random_state=0,
                                                n_jobs=-1)),
    ("AdaBoost (B=200)", AdaBoostClassifier(n_estimators=200, learning_rate=0.5,
                                            random_state=0)),
    ("SVM (RBF)", Pipeline([("sc", StandardScaler()), ("svc", SVC(C=1))])),
]

fig, axes = subplots(2, 4, figsize=(11, 5.6))
linhas = []
for ax, (nome, m) in zip(axes.ravel(), todos):
    m.fit(Xb, yb)
    tr, te = m.score(Xb, yb), m.score(Xb_te, yb_te)
    desenhar(ax, m, Xb, yb, f"{nome}\ntreino {tr:.3f}  teste {te:.3f}")
    linhas.append({"metodo": nome, "acuracia (treino)": tr, "acuracia (teste)": te,
                   "diferenca": tr - te})

In [ ]:
pd.DataFrame(linhas).set_index("metodo").sort_values("acuracia (teste)",
                                                     ascending=False).round(4)

Duas leituras, e as duas resumem o curso.

**A coluna `diferenca` é o otimismo do erro de treino, medido** — e ela desmente uma
crença comum. O KNN com $k=1$ e a árvore cheia acertam 100% do treino e ficam no
fundo da tabela de teste. Mas a **floresta também acerta 100% do treino**, com uma
diferença quase tão grande, e mesmo assim vai bem melhor que os dois.
>
Ou seja: *interpolar os dados de treino não é, por si só, um diagnóstico de nada.*
O que separa a floresta da árvore cheia não é quanto elas decoram, é o que sobra
depois — e o que sobra é a média sobre árvores descorrelacionadas (Aula 06, Seção
6). O único número que decide alguma coisa é a coluna do teste.

**A fronteira revela o método.** A logística só sabe traçar reta. O KNN produz
mosaicos, a árvore produz retângulos, a SVM com RBF produz curvas suaves, a floresta
produz uma escada tão fina que parece curva. São quatro maneiras diferentes de ser
flexível, e a escolha entre elas é sobre que **forma** você acredita que a fronteira
tem — que é a mesma conversa da Aula 05 sobre supor estrutura.

---
## 7. Caso real, com as métricas certas

Fechamos no `breast_cancer`, comparando os métodos deste bloco. E, aprendida a
lição da Aula 09, não só por acurácia.

In [ ]:
dados = load_breast_cancer()
X_tr, X_ts, y_tr, y_ts = skm.train_test_split(dados.data, dados.target,
                                              test_size=0.3, random_state=0,
                                              stratify=dados.target)

candidatos = {
    "logistica": Pipeline([("sc", StandardScaler()),
                           ("lg", LogisticRegression(max_iter=5000))]),
    "KNN (k por CV)": skm.GridSearchCV(
        Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier())]),
        {"knn__n_neighbors": [1, 3, 5, 9, 15, 25]}, cv=5, scoring="roc_auc"),
    "arvore podada": skm.GridSearchCV(
        DecisionTreeClassifier(random_state=0),
        {"ccp_alpha": np.linspace(0, 0.05, 20)}, cv=5, scoring="roc_auc"),
    "floresta": RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=0),
    "boosting": GradientBoostingClassifier(random_state=0),
    "SVM (RBF)": Pipeline([("sc", StandardScaler()), ("svc", SVC(C=10, probability=True,
                                                                random_state=0))]),
}

linhas = []
for nome, m in candidatos.items():
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_ts)[:, 1]
    linhas.append({"metodo": nome, "acuracia": (m.predict(X_ts) == y_ts).mean(),
                   "AUC": roc_auc_score(y_ts, p),
                   "AP": average_precision_score(y_ts, p)})
pd.DataFrame(linhas).set_index("metodo").sort_values("AUC", ascending=False).round(4)

Repare que a ordem por acurácia e a ordem por AUC **não são a mesma**. É a Aula 09
outra vez: acurácia mede uma decisão tomada no corte $0{,}5$; a AUC mede a qualidade
do ordenamento, independente de corte. Ao reportar um modelo, diga qual das duas
perguntas você respondeu.

E note o quanto os métodos estão próximos. Num problema com sinal forte e 30
covariáveis informativas, quase tudo funciona — o que é um bom lembrete final de
que a escolha do método costuma importar bem menos do que a qualidade dos dados, a
honestidade da validação e a adequação da métrica.

> **Sua vez.** Refaça a tabela usando o `bank_train_redux.csv` da Aula 09, que tem
> uma classe rara. As ordens por acurácia e por AP vão se separar muito mais. Depois
> escolha um corte por custo (Aula 09, Seção 5) e diga quantos clientes cada método
> mandaria investigar.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| KNN em classificação | §2 | é a mesma média, agora de indicadoras; com $k=5$ só há 6 probabilidades possíveis |
| impureza | §3 | Gini e entropia são estritamente côncavas; o erro de classificação não |
| o contraexemplo | §3 | pai $p=0{,}75$, filhos $0{,}50$ e $1{,}00$: erro reduz **zero**, Gini reduz $0{,}125$ |
| Gini × entropia | §4 | escolha de gosto: as árvores concordam em quase todo o teste |
| `max_features` | §5 | `RandomForestRegressor()` de fábrica **é bagging**, não floresta |
| interpolar ≠ superajustar | §6 | a floresta também acerta 100% do treino, e ainda assim vai bem |
| formas de fronteira | §6 | reta, mosaico, retângulos, curva suave — quatro maneiras de ser flexível |
| caso real | §7 | a ordem por acurácia e a ordem por AUC não coincidem |

**Leitura recomendada.** [AME] §8.3–8.6 (KNN, árvores e *ensembles* em
classificação). [ISLP] §4.7.6 (KNN como classificador) e §8.1.2, que traz exatamente
a discussão da Seção 3 sobre por que não se cresce árvore com o erro de
classificação. Vale reler também a §8.2 inteira com os olhos da Aula 06 — os
argumentos são os mesmos, com a impureza no lugar da soma de quadrados.

**Para praticar.** `Lista teorica 11.pdf` (teórica, com gabarito) e
`Lista prática 11.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** Fecha o curso regular. As três aulas extras saem do aprendizado
supervisionado: E1 e E2 tratam de problemas **sem $Y$** — agrupar e reduzir
dimensão —, e E3 aplica tudo a texto, que é onde a maldição da dimensionalidade da
Aula 05 aparece na sua forma mais pura.